# 📦 W6: F&B 재고 예측 & 발주 자동화
**hexa-2 — Week 6** | ⏱️ 60분 | 🎯 재고 부족 예측 + 발주 타이밍 + Gemini

## Step 0: 환경 설정

In [ ]:
import subprocess
subprocess.run(['apt-get','-qq','-y','install','fonts-nanum'],capture_output=True)
!pip install -q google-generativeai pandas matplotlib
import matplotlib.pyplot as plt, matplotlib.font_manager as fm, pandas as pd
_n=[f for f in fm.findSystemFonts() if 'Nanum' in f]
if _n: fm.fontManager.addfont(_n[0]); plt.rcParams['font.family']=fm.FontProperties(fname=_n[0]).get_name()
plt.rcParams['axes.unicode_minus']=False
try:
    from google.colab import userdata; API_KEY=userdata.get('GEMINI_API_KEY')
except: API_KEY=input('GEMINI_API_KEY: ')
import google.generativeai as genai; genai.configure(api_key=API_KEY)
model=genai.GenerativeModel('gemini-2.0-flash'); print('✅ 준비 완료')


## Step 1: 식자재 정보 입력 (✏️)

In [ ]:
import pandas as pd, numpy as np; np.random.seed(42)
INGREDIENTS=pd.DataFrame({
    '재료명':['돼지고기(kg)','두부(모)','배추김치(kg)','고추장(kg)','간장(L)'],
    '현재재고':[15,30,20,5,8],
    '일일소비':np.random.uniform(1,5,5).round(1),
    '최소안전재고':[10,20,15,3,5],
    '발주리드타임(일)':[2,1,3,5,7]
})
INGREDIENTS['예상소진일']=(INGREDIENTS['현재재고']/INGREDIENTS['일일소비']).round(0).astype(int)
INGREDIENTS['발주필요']=INGREDIENTS['예상소진일']<=INGREDIENTS['발주리드타임(일)']+1
print(INGREDIENTS.to_string(index=False))


## Step 2: 재고 현황 시각화

In [ ]:
fig,(ax1,ax2)=plt.subplots(1,2,figsize=(12,4))
clr=['#e74c3c' if r['발주필요'] else '#2ecc71' for _,r in INGREDIENTS.iterrows()]
ax1.barh(INGREDIENTS['재료명'],INGREDIENTS['현재재고'],color=clr)
ax1.set_title('현재 재고 현황 (빨강=발주 필요)')
ax2.bar(INGREDIENTS['재료명'],INGREDIENTS['예상소진일'],color='steelblue')
ax2.axhline(3,color='red',linestyle='--',label='위험선(3일)')
ax2.set_title('예상 소진일'); ax2.legend()
plt.tight_layout(); plt.savefig('inventory.png',dpi=150,bbox_inches='tight'); plt.show()


## Step 3: 발주 메시지 자동 생성

In [ ]:
urgent=INGREDIENTS[INGREDIENTS['발주필요']]
if len(urgent)>0:
    items=', '.join([f"{r['재료명']}({r['현재재고']}/{r['최소안전재고']})" for _,r in urgent.iterrows()])
    p=f"""식자재 발주 담당자로서 긴급 발주 요청 문자를 작성하세요.
재고 부족 재료: {items}
간결하고 명확하게, 100자 이내."""
    msg=model.generate_content(p)
    print(f'📱 발주 메시지:\n{msg.text}')
else:
    print('✅ 현재 발주 필요 없음')
    msg=type('obj',(),{'text':'발주 불필요'})()


## Step 4: Gemini 재고 최적화 전략

In [ ]:
p2=f"""F&B 재고 관리 전문가로서 최적화 전략:
발주 필요 항목: {len(urgent)}개/{len(INGREDIENTS)}개
재고 절감 방안 + 발주 자동화 로직 + 안전재고 기준 재설정. 마크다운."""
resp=model.generate_content(p2)
print(resp.text)


## Step 5: 다운로드

In [ ]:
INGREDIENTS.to_csv('inventory_report.csv',index=False,encoding='utf-8-sig')
with open('order_strategy.md','w',encoding='utf-8') as f: f.write(resp.text)
from google.colab import files
files.download('inventory_report.csv'); files.download('inventory.png')
print('✅ 완료!')
